<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/04_vision/40_convolucion_desde_cero.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Convolución y pooling desde cero

**Pregunta guía:** ¿Qué información local extrae un filtro?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Correlación usada por las CNN

Las bibliotecas suelen llamar convolución a la correlación cruzada
$$Y_{ij}=\sum_{u,v}X_{i+u,j+v}K_{uv}+b.$$
Con entrada $H\times W$, kernel $K_h\times K_w$, padding $P$ y stride
$S$, la altura de salida es
$\lfloor(H+2P-K_h)/S\rfloor+1$. Los pesos compartidos detectan el mismo
patrón en posiciones distintas.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

def correlación_2d(imagen, kernel, stride=1, padding=0):
    imagen = np.pad(imagen, padding)
    kh, kw = kernel.shape
    oh = (imagen.shape[0]-kh)//stride + 1
    ow = (imagen.shape[1]-kw)//stride + 1
    salida = np.empty((oh, ow), dtype=float)
    for i in range(oh):
        for j in range(ow):
            región = imagen[i*stride:i*stride+kh, j*stride:j*stride+kw]
            salida[i, j] = np.sum(región*kernel)
    return salida

def max_pool_2d(imagen, tamaño=2, stride=2):
    oh = (imagen.shape[0]-tamaño)//stride + 1
    ow = (imagen.shape[1]-tamaño)//stride + 1
    salida = np.empty((oh, ow))
    for i in range(oh):
        for j in range(ow):
            región = imagen[i*stride:i*stride+tamaño, j*stride:j*stride+tamaño]
            salida[i,j] = región.max()
    return salida


In [ ]:
digits = load_digits()
imagen = digits.images[np.flatnonzero(digits.target == 8)[0]]
kernels = {
    "Sobel horizontal": np.array([[-1,-2,-1],[0,0,0],[1,2,1]]),
    "Sobel vertical": np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    "Laplaciano": np.array([[0,1,0],[1,-4,1],[0,1,0]]),
}
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(imagen, cmap="gray"); axes[0].set_title("entrada")
for ax, (nombre, kernel) in zip(axes[1:4], kernels.items()):
    mapa = correlación_2d(imagen, kernel, padding=1)
    ax.imshow(mapa, cmap="coolwarm"); ax.set_title(nombre)
pooled = max_pool_2d(imagen)
axes[4].imshow(pooled, cmap="gray"); axes[4].set_title("max pooling")
for ax in axes: ax.axis("off")
plt.show()


## Varios canales y parámetros

Un filtro 3D abarca todos los canales de entrada. Con $C_{in}$ canales,
$C_{out}$ filtros y kernel $K_h\times K_w$, hay
$K_hK_wC_{in}C_{out}+C_{out}$ parámetros: no depende del tamaño espacial.
Pooling no aprende pesos; reduce resolución y aumenta el campo receptivo.


In [ ]:
def parámetros_conv(kh, kw, canales_entrada, filtros, bias=True):
    return kh*kw*canales_entrada*filtros + (filtros if bias else 0)

print("Conv 3×3, 3→64:", parámetros_conv(3,3,3,64), "parámetros")
print("Capa densa 224×224×3→64:", 224*224*3*64+64, "parámetros")


**Ejercicios:** compare correlación y convolución rotando el kernel;
implemente average pooling; calcule tamaños para stride 2; aplique los
filtros a una matriz que represente un campo escalar y discuta relación
con operadores diferenciales discretos.
